# AdaBoost from scratch — NumPy only, checked against scikit-learn

This notebook builds AdaBoost with nothing but NumPy, one README step at a time, and checks every step against
scikit-learn's `AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1))`. The goal is an **exact** match:
the same stumps, the same `estimator_weights_` and `estimator_errors_`, the same `decision_function`, the same predictions.

**How to use it**

1. Run the imports cell first, then top to bottom (`Run All` works). Each section only uses functions defined above it.
2. Every section has the same shape: the README step and its formula → the code → a small **check** (an assertion or a
   printed comparison). Sections marked **Implementation hurdle** show a real difficulty, demonstrated with a timing or a
   naive version that breaks.
3. The datasets are the CSVs written by [`adaboost_lab.ipynb`](adaboost_lab.ipynb) (`data/good_fit.csv`,
   `data/bad_fit.csv`); they are committed, so you do not have to run the lab first. No internet needed.

**Theory → code map**

| README section / equation | Function | Notebook section |
|---|---|---|
| §2.4 weighted Gini of a split, W_L·Gini_L + W_R·Gini_R | `split_costs_for_feature` | 1 |
| §2.2 decision stump; §3 step 2 “try every feature, keep the lowest Gini” | `presort_features`, `find_best_split`, `fit_stump`, `predict_stump` | 1 |
| §3 step 3, weighted error ε = Σ wᵢ·[wrong] | inside `fit_adaboost_textbook`, `fit_adaboost_samme` | 2, 3 |
| §3 steps 4–6, α = ½ ln((1−ε)/ε), w ← w·e^(−α y h), normalise | `fit_adaboost_textbook` | 2 |
| §3 final rule, sign(Σ α_m h_m(x)) | `score_textbook`, `predict_textbook` | 2 |
| §4.6 SAMME: α = η·(ln((1−ε)/ε) + ln(K−1)), only wrong points × e^α | `fit_adaboost_samme` | 3 |
| §6.3 `decision_function` = Σ α·(+1 or −1/(K−1)) / Σ α | `decision_function_samme`, `predict_samme` | 4 |
| §4.4, §8.1 numerical stability of the weights | `run_unnormalised_weights`, `fit_adaboost_samme_logspace` | 3 |

| Section | What it builds | Implementation hurdle shown |
|---|---|---|
| 1 | the weighted stump | split-search cost (naive vs sort + cumulative sums vs pre-sorted); ties, midpoints and float32; Gini vs misclassification error |
| 2 | textbook AdaBoost (labels ±1) | — (reproduces the README hand example exactly) |
| 3 | SAMME, scikit-learn's version | ε = 0 and ε ≥ 1 − 1/K; weights overflowing / tiny weights and scikit-learn's clipping |
| 4 | decision function and predictions | — |
| 5 | comparison with scikit-learn on both CSVs | exactness, speed |

In [1]:
import time
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 80, "axes.grid": True, "grid.alpha": 0.3})


def sklearn_adaboost(n_estimators, learning_rate=1.0):
    """The library model every function here is compared with (stumps, fixed seed)."""
    return AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=n_estimators,
                              learning_rate=learning_rate, random_state=0)

## 0 · The data

The two CSVs from the lab (2 features, labels 0/1, 1000 rows each; the bad-fit file has 20 % of the labels flipped). We use
the lab's split: 70 % train / 30 % test with `random_state=0`.

In [2]:
DATA = Path("data")
datasets = {}
for name in ["good_fit", "bad_fit"]:
    table = np.loadtxt(DATA / f"{name}.csv", delimiter=",", skiprows=1)
    X, y = table[:, :2], table[:, 2].astype(int)
    idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.3, random_state=0)
    datasets[name] = (X[idx_train], y[idx_train], X[idx_test], y[idx_test])
    print(f"{name}: X {X.shape}, classes {np.unique(y)}, share of class 1 = {y.mean():.3f}, train/test = {len(idx_train)}/{len(idx_test)}")

Xtr, ytr, Xte, yte = datasets["good_fit"]

good_fit: X (1000, 2), classes [0 1], share of class 1 = 0.496, train/test = 700/300
bad_fit: X (1000, 2), classes [0 1], share of class 1 = 0.530, train/test = 700/300


---
## 1 · The weighted decision stump (README §2.2–2.4)

A stump asks one question, “is feature *j* ≤ *t*?”, and predicts one class on each side. AdaBoost hands it a **weight**
per training point, so everything a normal tree counts, the stump **sums weights** instead:

* weight on each side: W_L, W_R; weight of class *c* on each side: S_L,c, S_R,c;
* Gini impurity of a side: Gini = 1 − Σ_c (S_c / W)²;
* the **cost** of a cut: W_L·Gini_L + W_R·Gini_R (smaller = purer sides). Every feature and every cut between two
  neighbouring distinct values is tried; the cheapest wins (README §3 step 2: “a stump is created for each feature and the one
  with minimum Gini is picked”);
* each side then predicts its heaviest class.

**The trick:** sort the points by the feature once; then the left side of the *k*-th cut is just the first *k* sorted points,
so W_L and S_L for **all** cuts at once are **cumulative sums** (`np.cumsum`). And since boosting changes only the
*weights*, never X, the sort order can be computed **once** and reused in every round (`presort_features`).

Details copied from scikit-learn so the results match bit for bit (each one is shown in the hurdles below):
features are cast to `float32`; the threshold is the midpoint `a/2 + b/2` of two neighbouring sorted values; cuts between equal
values (closer than 1e-7) are not allowed; among equally cheap cuts the first one (lowest feature, then lowest threshold) is kept.

In [3]:
def presort_features(X32):
    """Sort order of every column, computed once (boosting changes the weights, never X)."""
    return np.argsort(X32, axis=0, kind="stable")


def split_costs_for_feature(x_sorted, w_sorted, onehot_sorted, W_node, class_totals):
    """W_L*Gini_L + W_R*Gini_R for every cut between neighbouring sorted values (inf where the cut is impossible)."""
    WL = np.cumsum(w_sorted)[:-1]                                    # weight left of each cut
    SL = np.cumsum(w_sorted[:, None] * onehot_sorted, axis=0)[:-1]   # weight of each class left of each cut
    WR, SR = W_node - WL, class_totals - SL
    sq_left, sq_right = np.zeros_like(WL), np.zeros_like(WL)
    for c in range(SL.shape[1]):                                     # same summation order as scikit-learn
        sq_left = sq_left + SL[:, c] * SL[:, c]
        sq_right = sq_right + SR[:, c] * SR[:, c]
    cost = WL * (1.0 - sq_left / (WL * WL)) + WR * (1.0 - sq_right / (WR * WR))
    x64 = x_sorted.astype(np.float64)
    can_cut = x64[1:] > x64[:-1] + 1e-7                              # never cut between (nearly) equal values
    can_cut &= ~np.isnan(cost)               # 0/0 when one side's weight rounds to 0: never chosen (as in scikit-learn)
    return np.where(can_cut, cost, np.inf)


def find_best_split(X32, order, y_idx, w, n_classes):
    """Cheapest (feature, threshold) over all features: the weighted-Gini rule of scikit-learn's tree."""
    onehot = np.eye(n_classes)[y_idx]
    W_node = np.cumsum(w)[-1]                                        # sequential sums, like scikit-learn
    class_totals = np.cumsum(w[:, None] * onehot, axis=0)[-1]
    best = {"cost": np.inf, "feature": None, "threshold": None}
    for j in range(X32.shape[1]):
        o = order[:, j]
        costs = split_costs_for_feature(X32[o, j], w[o], onehot[o], W_node, class_totals)
        k = int(np.argmin(costs))                                    # first of equal minima
        if costs[k] < best["cost"]:                                  # strictly better: earlier feature wins ties
            a, b = np.float64(X32[o[k], j]), np.float64(X32[o[k + 1], j])
            best = {"cost": costs[k], "feature": j, "threshold": a / 2.0 + b / 2.0}
    return best


def fit_stump(X32, order, y_idx, w, n_classes, split_finder=find_best_split):
    """A one-question tree: the best split, and the heaviest class on each side."""
    split = split_finder(X32, order, y_idx, w, n_classes)
    left = X32[:, split["feature"]].astype(np.float64) <= split["threshold"]
    left_mass = np.bincount(y_idx[left], weights=w[left], minlength=n_classes)
    right_mass = np.bincount(y_idx[~left], weights=w[~left], minlength=n_classes)
    return {"feature": split["feature"], "threshold": split["threshold"],
            "left_class": int(np.argmax(left_mass)), "right_class": int(np.argmax(right_mass))}


def predict_stump(stump, X32):
    """Class index for every row: left_class if x[feature] <= threshold, else right_class."""
    left = X32[:, stump["feature"]].astype(np.float64) <= stump["threshold"]
    return np.where(left, stump["left_class"], stump["right_class"])

**Check:** our stump vs `DecisionTreeClassifier(max_depth=1).fit(X, y, sample_weight=w)` for 50 random weight vectors on
the good-fit training set: same feature, same threshold (exactly, `==`), same predictions on the test set.

In [4]:
rng = np.random.default_rng(0)
X32_tr, X32_te = Xtr.astype(np.float32), Xte.astype(np.float32)
order_tr = presort_features(X32_tr)
same_feature = same_threshold = same_predictions = 0
for trial in range(50):
    w = rng.exponential(size=len(ytr)) ** 3                      # very uneven weights, like late boosting rounds
    w /= w.sum()
    ours = fit_stump(X32_tr, order_tr, ytr, w, 2)
    tree = DecisionTreeClassifier(max_depth=1, random_state=0).fit(Xtr, ytr, sample_weight=w)
    same_feature += ours["feature"] == tree.tree_.feature[0]
    same_threshold += ours["threshold"] == tree.tree_.threshold[0]
    same_predictions += np.array_equal(predict_stump(ours, X32_te), tree.predict(Xte))
print(f"out of 50 random weightings: same feature {same_feature}, identical threshold {same_threshold}, identical test predictions {same_predictions}")
assert same_feature == same_threshold == same_predictions == 50

out of 50 random weightings: same feature 50, identical threshold 50, identical test predictions 50


### Implementation hurdle 1 · The cost of searching for the split

A stump must look at every possible cut of every feature. Three ways to do it:

* **naive:** for each candidate threshold, re-scan all *n* points to sum the weights on each side → about *n* cuts × *n*
  points = O(n²) per feature;
* **sort + cumulative sums:** sort once per call (O(n log n)), then one `cumsum` gives every cut's sums (O(n));
* **pre-sorted:** as above but the sort is done once for the whole boosting run, so each round is just O(n).

All three must pick the same split; we time one call of each on the 700 training points.

In [5]:
def find_best_split_naive(X32, order, y_idx, w, n_classes):
    """Same rule, no tricks: for every distinct midpoint, recompute both sides' weights from scratch."""
    best = {"cost": np.inf, "feature": None, "threshold": None}
    for j in range(X32.shape[1]):
        values = np.unique(X32[:, j]).astype(np.float64)
        for a, b in zip(values[:-1], values[1:]):
            t = a / 2.0 + b / 2.0
            left = X32[:, j].astype(np.float64) <= t
            cost = 0.0
            for side in (left, ~left):
                W = w[side].sum()
                S = np.bincount(y_idx[side], weights=w[side], minlength=n_classes)
                cost += W * (1.0 - np.sum(S ** 2) / W ** 2)
            if cost < best["cost"] - 1e-15:
                best = {"cost": cost, "feature": j, "threshold": t}
    return best


def find_best_split_sort_each_time(X32, order, y_idx, w, n_classes):
    """Sort + cumulative sums, but sorting again in every call (ignores the pre-computed order)."""
    return find_best_split(X32, presort_features(X32), y_idx, w, n_classes)


w = rng.exponential(size=len(ytr)); w /= w.sum()
for name, finder, repeats in [("naive O(n^2)", find_best_split_naive, 1),
                              ("sort + cumsum each call", find_best_split_sort_each_time, 50),
                              ("pre-sorted + cumsum", find_best_split, 50)]:
    t0 = time.perf_counter()
    for _ in range(repeats):
        split = finder(X32_tr, order_tr, ytr, w, 2)
    seconds = (time.perf_counter() - t0) / repeats
    print(f"{name:24s}: feature {split['feature']}, threshold {split['threshold']:+.4f}, "
          f"{seconds * 1e3:8.3f} ms per stump -> {seconds * 400:7.2f} s for 400 rounds")

naive O(n^2)            : feature 1, threshold -0.6335,   16.296 ms per stump ->    6.52 s for 400 rounds
sort + cumsum each call : feature 1, threshold -0.6335,    0.093 ms per stump ->    0.04 s for 400 rounds
pre-sorted + cumsum     : feature 1, threshold -0.6335,    0.076 ms per stump ->    0.03 s for 400 rounds


> **Implementation hurdle — what this shows**
> * All three find the same split. The naive search takes about 18 ms per stump, i.e. about 7 s for a 400-round model on just
>   700 points, and it grows with n²: 10× more data → 100× slower.
> * Sort + cumulative sums brings that down to about 0.1 ms — roughly 200× faster; pre-sorting once also saves the sort in
>   every round (about 0.08 ms). That is why our whole 400-round fit in section 5 takes about 0.04 s. (Timings vary a little
>   from run to run and by machine.)
> * scikit-learn's tree uses the same cumulative-sum idea in compiled code but sorts again for every tree it builds,
>   because a general tree cannot know that the next tree will see the same X.

### Implementation hurdle 2 · Ties, midpoints and float32

The data were rounded to 3 decimals, so some values repeat. A cut **between two equal values** is impossible (both points
would have to go to the same side), so such positions get cost ∞. Thresholds are midpoints of neighbouring *distinct* values.
And scikit-learn's trees store X as `float32`: the midpoint of two float32 numbers is not the same float64 number as the
midpoint of the original float64 numbers. We count how often that matters.

In [6]:
for j in range(2):
    print(f"feature x{j + 1}: {len(Xtr)} training values, {len(np.unique(Xtr[:, j]))} distinct -> "
          f"{len(Xtr) - len(np.unique(Xtr[:, j]))} positions where no cut is allowed")

sk_400 = sklearn_adaboost(400).fit(Xtr, ytr)
sk_thresholds = np.array([est.tree_.threshold[0] for est in sk_400.estimators_])
sk_features = np.array([est.tree_.feature[0] for est in sk_400.estimators_])
thr_from_float64 = []
for f, t in zip(sk_features, sk_thresholds):
    values = np.unique(Xtr[:, f])                                    # the same cut, computed from float64 data
    k = np.searchsorted(values, t)
    thr_from_float64.append(values[k - 1] / 2 + values[k] / 2)
thr_from_float64 = np.array(thr_from_float64)
moved = sk_thresholds != thr_from_float64
print(f"\n400 thresholds: {moved.sum()} differ (bit for bit) if computed from float64 values; "
      f"largest difference {np.abs(sk_thresholds - thr_from_float64).max():.1e}")
# scikit-learn compares float32 test values with its threshold; a float64 implementation would compare float64 with float64
changed = [np.flatnonzero((X32_te[:, f].astype(np.float64) <= a) != (Xte[:, f] <= b))
           for f, a, b in zip(sk_features, sk_thresholds, thr_from_float64)]
n_changed = sum(len(c) for c in changed)
print(f"(test point, cut) pairs where the two versions send the point to different sides: {n_changed}")
example = next((f, a, b, i) for f, a, b, c in zip(sk_features, sk_thresholds, thr_from_float64, changed) for i in c)
f, a, b, i = example
print(f"example: x{f + 1} = {Xte[i, f]} lies exactly halfway between two training values; "
      f"float32 cut {a!r} vs float64 cut {b!r}; float32(x) = {np.float64(X32_te[i, f])!r}")

feature x1: 700 training values, 593 distinct -> 107 positions where no cut is allowed
feature x2: 700 training values, 601 distinct -> 99 positions where no cut is allowed



400 thresholds: 400 differ (bit for bit) if computed from float64 values; largest difference 2.1e-08
(test point, cut) pairs where the two versions send the point to different sides: 4
example: x1 = -0.604 lies exactly halfway between two training values; float32 cut np.float64(-0.6040000021457672) vs float64 cut np.float64(-0.604); float32(x) = np.float64(-0.6039999723434448)


> **Implementation hurdle — what this shows**
> * Repeated values are common here (x1 has 107 repeated positions among 700 values, x2 has 99), so skipping cuts between equal
>   values is not a corner case. Our `can_cut` mask does it, with scikit-learn's tolerance of 1e-7.
> * All 400 thresholds would differ in the last bits if we took midpoints of the float64 values (by at most 2.1e-08). Usually that
>   changes nothing — but 4 times a test point lies **exactly** on a cut (e.g. x1 = −0.604, halfway between the training values
>   −0.605 and −0.603). In float32, −0.604 becomes −0.60399997…, which is *above* the float32 cut −0.60400000…; in float64 it is
>   *equal* to the cut −0.604 and goes left. So a float64 implementation would send that point to the other side. Matching
>   scikit-learn exactly needs the float32 cast for both training and prediction.

### Implementation hurdle 3 · Gini is not the same as “fewest weighted mistakes”

AdaBoost's theory (README §4.3) says each round should pick the stump with the **smallest weighted error** ε. scikit-learn's
tree instead picks the cut with the smallest **weighted Gini** (what the old notes said: “the one with minimum
Gini/entropy is picked”). The two rules usually agree, but not always. We replay scikit-learn's 400 rounds on the good-fit
data and, at every round, compare the Gini choice with the error choice on the same weights.

In [7]:
def find_best_split_by_error(X32, order, y_idx, w, n_classes):
    """The rule AdaBoost's derivation asks for: the cut whose sides' heaviest classes make the fewest weighted mistakes."""
    onehot = np.eye(n_classes)[y_idx]
    W_node, class_totals = w.sum(), (w[:, None] * onehot).sum(axis=0)
    best = {"cost": np.inf, "feature": None, "threshold": None}
    for j in range(X32.shape[1]):
        o = order[:, j]
        WL = np.cumsum(w[o])[:-1]
        SL = np.cumsum(w[o][:, None] * onehot[o], axis=0)[:-1]
        error = (WL - SL.max(axis=1)) + ((W_node - WL) - (class_totals - SL).max(axis=1))
        x64 = X32[o, j].astype(np.float64)
        error = np.where(x64[1:] > x64[:-1] + 1e-7, error, np.inf)
        k = int(np.argmin(error))
        if error[k] < best["cost"]:
            best = {"cost": error[k], "feature": j,
                    "threshold": np.float64(X32[o[k], j]) / 2.0 + np.float64(X32[o[k + 1], j]) / 2.0}
    return best


def weights_used_by_sklearn(model, X, y):
    """Replay the weights scikit-learn used in each round (clip, multiply the wrong ones by e^alpha, normalise)."""
    w, used = np.full(len(y), 1 / len(y)), []
    for est, alpha in zip(model.estimators_, model.estimator_weights_):
        w = np.clip(w, np.finfo(float).eps, None)
        used.append(w)
        w = np.exp(np.log(w) + alpha * (est.predict(X) != y))
        w = w / w.sum()
    return used


differ = []
for m, w in enumerate(weights_used_by_sklearn(sk_400, Xtr, ytr)):
    by_gini = fit_stump(X32_tr, order_tr, ytr, w, 2)
    by_error = fit_stump(X32_tr, order_tr, ytr, w, 2, split_finder=find_best_split_by_error)
    if (by_gini["feature"], by_gini["threshold"]) != (by_error["feature"], by_error["threshold"]):
        eps_gini = np.sum(w[predict_stump(by_gini, X32_tr) != ytr])
        eps_error = np.sum(w[predict_stump(by_error, X32_tr) != ytr])
        differ.append((m + 1, by_gini, eps_gini, by_error, eps_error))
print(f"rounds where the Gini stump and the min-error stump differ: {len(differ)} of 400")
m, g, eg, e, ee = differ[0]
print(f"first one, round {m}:")
print(f"   Gini rule  : x{g['feature'] + 1} <= {g['threshold']:+.4f}  weighted error {eg:.4f}   (scikit-learn's choice: "
      f"x{sk_features[m - 1] + 1} <= {sk_thresholds[m - 1]:+.4f})")
print(f"   error rule : x{e['feature'] + 1} <= {e['threshold']:+.4f}  weighted error {ee:.4f}")
print(f"mean extra weighted error of the Gini stump over those rounds: {np.mean([d[2] - d[4] for d in differ]):.4f}")

rounds where the Gini stump and the min-error stump differ: 373 of 400
first one, round 1:
   Gini rule  : x1 <= -0.6840  weighted error 0.3843   (scikit-learn's choice: x1 <= -0.6840)
   error rule : x1 <= -0.4595  weighted error 0.3671
mean extra weighted error of the Gini stump over those rounds: 0.0294


> **Implementation hurdle — what this shows**
> * In 373 of 400 rounds the two rules pick different stumps — already in round 1: the Gini cut `x1 ≤ −0.684` has weighted
>   error 0.3843, the min-error cut `x1 ≤ −0.4595` only 0.3671. On average the Gini stump makes 0.029 more weighted error in the
>   rounds where they differ. Gini prefers cuts that make one side very *pure* (here: a thin, almost all-blue strip at the left
>   edge) even at the price of a few more weighted mistakes overall.
> * scikit-learn's choice is always the Gini one, so an implementation that picks by error (the textbook rule) is a *valid*
>   AdaBoost but **cannot** reproduce `AdaBoostClassifier` — that is why `split_costs_for_feature` uses Gini.
> * Both work: section 5 fits a full model with each rule and compares test accuracy.

---
## 2 · Textbook AdaBoost, and the hand example (README §3, §4.3, §5)

Labels are written as y ∈ {−1, +1} and a stump's output as h(x) ∈ {−1, +1}. For m = 1 … M:

1. fit the stump to the current weights w (start: all 1/N);
2. weighted error ε_m = Σ wᵢ over the misclassified points (weights sum to 1);
3. stop if ε_m = 0 (perfect) or ε_m ≥ ½ (no better than a coin);
4. stage value α_m = ½ ln((1 − ε_m)/ε_m);
5. wᵢ ← wᵢ · e^(−α_m yᵢ h_m(xᵢ)) (× e^(α) if wrong, × e^(−α) if right), then divide by the sum.

Prediction: F(x) = Σ α_m h_m(x), class = sign(F(x)).

In [8]:
def fit_adaboost_textbook(X, y, n_rounds):
    """Binary AdaBoost with the ±1 update and alpha = 1/2 ln((1-eps)/eps). Returns a flat dict of arrays."""
    classes, y_idx = np.unique(y, return_inverse=True)
    y_pm = 2 * y_idx - 1
    X32 = X.astype(np.float32); order = presort_features(X32)
    w = np.full(len(y), 1 / len(y))
    stumps, alphas, errors, weights_used = [], [], [], []
    for m in range(n_rounds):
        stump = fit_stump(X32, order, y_idx, w, 2)
        h = 2 * predict_stump(stump, X32) - 1
        eps = np.sum(w[h != y_pm])
        if eps <= 0 or eps >= 0.5:
            break
        alpha = 0.5 * np.log((1 - eps) / eps)
        stumps.append(stump); alphas.append(alpha); errors.append(eps); weights_used.append(w)
        w = w * np.exp(-alpha * y_pm * h)
        w = w / w.sum()
    return {"classes": classes, "stumps": stumps, "alphas": np.array(alphas),
            "errors": np.array(errors), "weights_used": np.array(weights_used)}


def score_textbook(model, X):
    """F(x) = sum_m alpha_m h_m(x) with h in {-1, +1}."""
    X32 = X.astype(np.float32)
    return sum(a * (2 * predict_stump(s, X32) - 1) for s, a in zip(model["stumps"], model["alphas"]))


def predict_textbook(model, X):
    return model["classes"][(score_textbook(model, X) > 0).astype(int)]

**Check — the README §5 example**, number for number: x = 1 … 8, labels `+ + + + − − + −`.

In [9]:
x_hand = np.arange(1, 9, dtype=float).reshape(-1, 1)
y_hand = np.array([1, 1, 1, 1, -1, -1, 1, -1])
hand = fit_adaboost_textbook(x_hand, y_hand, 3)

for m in range(3):
    s = hand["stumps"][m]
    print(f"round {m + 1}: weights x 168 = {(hand['weights_used'][m] * 168).round(6)}")
    print(f"          stump x <= {s['threshold']} -> {hand['classes'][s['left_class']]:+d}, eps = {hand['errors'][m]:.6f}, alpha = {hand['alphas'][m]:.6f}")
F_hand = score_textbook(hand, x_hand)
print("F(x) =", F_hand.round(4), "  predictions:", predict_textbook(hand, x_hand))

assert [s["threshold"] for s in hand["stumps"]] == [4.5, 7.5, 6.5]
assert np.allclose(hand["errors"], [1 / 8, 1 / 7, 5 / 24], rtol=0, atol=1e-15)
assert np.allclose(hand["alphas"], 0.5 * np.log([7, 6, 19 / 5]), rtol=0, atol=1e-15)
assert np.allclose(hand["weights_used"][2], np.array([1, 1, 1, 1, 6, 6, 7, 1]) / 24, rtol=0, atol=1e-15)
assert np.array_equal(predict_textbook(hand, x_hand), y_hand)
print("all README §5 numbers reproduced (to 1e-15)")

round 1: weights x 168 = [21. 21. 21. 21. 21. 21. 21. 21.]
          stump x <= 4.5 -> +1, eps = 0.125000, alpha = 0.972955
round 2: weights x 168 = [12. 12. 12. 12. 12. 12. 84. 12.]
          stump x <= 7.5 -> +1, eps = 0.142857, alpha = 0.895880
round 3: weights x 168 = [ 7.  7.  7.  7. 42. 42. 49.  7.]
          stump x <= 6.5 -> -1, eps = 0.208333, alpha = 0.667501
F(x) = [ 1.201  1.201  1.201  1.201 -0.745 -0.745  0.59  -1.201]   predictions: [ 1  1  1  1 -1 -1  1 -1]
all README §5 numbers reproduced (to 1e-15)


> **What this shows:** every number of the README table — weights 1/8 → (1/14, 7/14) → (1/24, 6/24, 7/24) (× 168 they are
> 21; 12 and 84; 7, 42 and 49), ε = 1/8, 1/7, 5/24, α = ½ ln 7, ½ ln 6, ½ ln(19/5), and F(x) = 1.201, −0.745, 0.590, −1.201 —
> comes out of the code to 15 decimals, and all eight points are classified correctly.

---
## 3 · SAMME: what scikit-learn actually computes (README §4.6)

scikit-learn (1.6 and later; only SAMME remains) differs from the textbook in three visible ways:

1. α_m = η · (ln((1 − ε_m)/ε_m) + ln(K − 1)), with η = `learning_rate` and K classes. For K = 2 and η = 1 that is **2× the
   textbook α**.
2. Only the **misclassified** points are multiplied, by e^(α_m); the correct ones are left alone. Then everything is normalised.
3. It stops when ε = 0 (keeps that learner with weight 1) or when ε ≥ 1 − 1/K (throws that learner away; error if it was the
   first). Before every round, weights below machine epsilon (2.2e-16) are raised to it. In the last round the weights are not
   updated.

Why 1 and 2 give the **same weights** as the textbook: the textbook multiplies by e^(−α y h), which is e^(−α) for right and
e^(+α) for wrong. Multiply *every* point by the same number e^(α) — normalising removes any common factor — and you get 1 for
right and e^(2α) for wrong: exactly SAMME's update with α_SAMME = 2α. And sign(Σ 2α h) = sign(Σ α h), so the predictions are
the same too.

In [10]:
def fit_adaboost_samme(X, y, n_estimators=50, learning_rate=1.0, split_finder=find_best_split, clip_tiny_weights=True):
    """scikit-learn's AdaBoostClassifier (SAMME) step by step. Returns a flat dict of arrays."""
    classes, y_idx = np.unique(y, return_inverse=True)
    K = len(classes)
    X32 = X.astype(np.float32); order = presort_features(X32)
    w = np.full(len(y), 1.0); w /= w.sum()
    tiny = np.finfo(np.float64).eps
    stumps, alphas, errors = [], [], []
    for m in range(n_estimators):
        if clip_tiny_weights:
            w = np.clip(w, tiny, None)
        stump = fit_stump(X32, order, y_idx, w, K, split_finder)
        wrong = predict_stump(stump, X32) != y_idx
        eps = np.mean(np.average(wrong, weights=w, axis=0))
        if eps <= 0:                                      # perfect learner: keep it with weight 1 and stop
            stumps.append(stump); alphas.append(1.0); errors.append(0.0)
            break
        if eps >= 1 - 1 / K:                              # no better than random guessing: discard and stop
            if not stumps:
                raise ValueError("first learner is no better than random guessing; nothing to boost")
            break
        alpha = learning_rate * (np.log((1 - eps) / eps) + np.log(K - 1))
        stumps.append(stump); alphas.append(alpha); errors.append(eps)
        if m == n_estimators - 1:                         # last round: no update needed
            break
        w = np.exp(np.log(w) + alpha * wrong)             # only the wrong ones grow
        w = w / np.sum(w)
    return {"classes": classes, "stumps": stumps, "alphas": np.array(alphas), "errors": np.array(errors)}

**Check:** on the hand example, SAMME's α is exactly 2× the textbook α, the weights it uses in every round are the same, and
scikit-learn agrees with both.

In [11]:
samme_hand = fit_adaboost_samme(x_hand, y_hand, 3)
sk_hand = sklearn_adaboost(3).fit(x_hand, y_hand)
print("textbook alpha x 2:", 2 * hand["alphas"])
print("our SAMME alpha   :", samme_hand["alphas"])
print("sklearn weights   :", sk_hand.estimator_weights_)
w_samme = weights_used_by_sklearn(sk_hand, x_hand, y_hand)
print("max |weights(textbook) - weights(SAMME)| over the 3 rounds:", max(np.abs(a - b).max() for a, b in zip(hand["weights_used"], w_samme)))
assert np.allclose(samme_hand["alphas"], 2 * hand["alphas"], rtol=0, atol=1e-14)
assert np.array_equal(samme_hand["alphas"], sk_hand.estimator_weights_)
assert np.array_equal(samme_hand["errors"], sk_hand.estimator_errors_)

textbook alpha x 2: [1.946 1.792 1.335]
our SAMME alpha   : [1.946 1.792 1.335]
sklearn weights   : [1.946 1.792 1.335]
max |weights(textbook) - weights(SAMME)| over the 3 rounds: 1.1102230246251565e-16


### Implementation hurdle 4 · ε = 0 and ε ≥ 1 − 1/K

Two edge cases end boosting. **ε = 0:** the formula would need ln(1/0) = ∞. **ε ≥ 1 − 1/K:** α would be ≤ 0 (the learner is no
better than guessing). We trigger both and check that we behave exactly like scikit-learn.

In [12]:
x_sep = np.arange(10, dtype=float).reshape(-1, 1)
y_sep = (x_sep[:, 0] > 4.5).astype(int)                               # one cut separates the classes perfectly
ours_sep = fit_adaboost_samme(x_sep, y_sep, 50)
sk_sep = sklearn_adaboost(50).fit(x_sep, y_sep)
print(f"separable data: ours {len(ours_sep['stumps'])} stump, alpha {ours_sep['alphas']}, error {ours_sep['errors']};  "
      f"sklearn {len(sk_sep.estimators_)} stump, alpha {sk_sep.estimator_weights_[:1]}, error {sk_sep.estimator_errors_[:1]}")

corners = np.array([[-1, -1], [-1, 1], [1, -1], [1, 1]], dtype=float)
offsets = np.random.default_rng(0).uniform(-0.5, 0.5, size=(25, 2))
X_xor, y_xor = np.vstack([c + offsets for c in corners]), np.repeat([0, 1, 1, 0], 25)
for name, fit in [("ours", lambda: fit_adaboost_samme(X_xor, y_xor, 50)), ("sklearn", lambda: sklearn_adaboost(50).fit(X_xor, y_xor))]:
    try:
        fit()
        print(f"XOR, {name}: fitted (unexpected)")
    except ValueError as err:
        print(f"XOR, {name}: ValueError -> {err}")

separable data: ours 1 stump, alpha [1.], error [0.];  sklearn 1 stump, alpha [1.], error [0.]
XOR, ours: ValueError -> first learner is no better than random guessing; nothing to boost
XOR, sklearn: ValueError -> BaseClassifier in AdaBoostClassifier ensemble is worse than random, ensemble can not be fit.


> **Implementation hurdle — what this shows**
> * Perfectly separable data: one stump, ε = 0. Instead of α = ∞ both implementations keep that stump with weight 1.0 and stop;
>   one perfect learner is the whole model.
> * Symmetric XOR: every stump has ε = 0.5 = 1 − 1/2 already in round 1, so there is nothing to boost; both raise a
>   `ValueError`. (Mid-run, the same rule simply stops boosting and discards the useless learner.)

### Implementation hurdle 5 · Weights that overflow, and weights that get too small

Normalising every round looks like a cosmetic step. It is not.

* **Without normalising,** SAMME's update keeps multiplying the weights of hard points by e^α. Their weights grow
  exponentially until they overflow to `inf`, and then ε becomes `nan` (∞/∞). (The textbook ± update has the opposite
  problem: the *sum* of the weights shrinks by the factor 2√(ε(1−ε)) every round — by exactly the training-error bound of
  README §4.4.)
* **With normalising,** easy points can still get weights far below 1e-16 after hundreds of rounds. scikit-learn raises
  every weight to at least machine epsilon (2.2e-16) before each round; without that step our run drifts from
  scikit-learn's.
* A third option is to keep **log-weights** and normalise with the log-sum-exp trick; that never overflows.

In [13]:
def run_unnormalised_weights(X, y, n_rounds, update):
    """Boost stumps WITHOUT normalising the weights. update='samme' (wrong x e^alpha) or 'textbook' (x e^(-alpha*y*h))."""
    classes, y_idx = np.unique(y, return_inverse=True)
    X32 = X.astype(np.float32); order = presort_features(X32)
    w = np.full(len(y), 1 / len(y))
    largest, total, epsilons = [], [], []
    for m in range(n_rounds):
        if not np.isfinite(np.sum(w)):                         # the weights (or their sum) overflowed: eps = inf/inf = nan
            epsilons.append(np.nan)
            break
        stump = fit_stump(X32, order, y_idx, w / w.sum(), 2)   # the stump itself only sees relative weights
        wrong = predict_stump(stump, X32) != y_idx
        eps = np.sum(w[wrong]) / np.sum(w)
        epsilons.append(eps)
        if update == "samme":
            w = w * np.exp(np.log((1 - eps) / eps) * wrong)
        else:
            a = 0.5 * np.log((1 - eps) / eps)
            w = w * np.exp(np.where(wrong, a, -a))
        largest.append(w.max()); total.append(w.sum())
    return {"largest": np.array(largest), "total": np.array(total), "epsilons": np.array(epsilons)}


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    unnorm = run_unnormalised_weights(Xtr, ytr, 20_000, update="samme")
finite = unnorm["largest"][np.isfinite(unnorm["largest"])]
print(f"SAMME update without normalising: largest weight after 1000 rounds = {unnorm['largest'][999]:.3e}, "
      f"after 10000 = {unnorm['largest'][9999]:.3e}")
if np.isnan(unnorm["epsilons"][-1]):
    print(f"   round {len(unnorm['epsilons'])}: the sum of the weights is inf, so eps = inf/inf = nan  "
          f"(largest single weight just before: {finite[-1]:.3e})")
else:
    print("   no overflow within the rounds run")
print(f"   NumPy warnings raised along the way: {sorted({str(c.message) for c in caught})}")

textbook_unnorm = run_unnormalised_weights(Xtr, ytr, 400, update="textbook")
bound = np.cumprod(2 * np.sqrt(textbook_unnorm["epsilons"] * (1 - textbook_unnorm["epsilons"])))
print(f"\ntextbook update without normalising: sum of weights after 400 rounds = {textbook_unnorm['total'][-1]:.6f}; "
      f"bound prod 2 sqrt(eps(1-eps)) = {bound[-1]:.6f}; max |difference| over all rounds = {np.abs(textbook_unnorm['total'] - bound).max():.1e}")

SAMME update without normalising: largest weight after 1000 rounds = 8.197e+28, after 10000 = 4.368e+189
   round 16574: the sum of the weights is inf, so eps = inf/inf = nan  (largest single weight just before: 1.258e+307)
   NumPy warnings raised along the way: ['divide by zero encountered in divide', 'invalid value encountered in divide', 'invalid value encountered in multiply', 'overflow encountered in reduce']

textbook update without normalising: sum of weights after 400 rounds = 0.052061; bound prod 2 sqrt(eps(1-eps)) = 0.052061; max |difference| over all rounds = 3.3e-16


Now the two fixes. `fit_adaboost_samme_logspace` keeps log-weights and normalises them with log-sum-exp; and we run our
normal `fit_adaboost_samme` with and without scikit-learn's clipping of tiny weights for 2000 rounds.

In [14]:
def fit_adaboost_samme_logspace(X, y, n_estimators=50, learning_rate=1.0):
    """SAMME with log-weights: log w += alpha*[wrong]; normalise with log-sum-exp. Cannot overflow."""
    classes, y_idx = np.unique(y, return_inverse=True)
    K = len(classes)
    X32 = X.astype(np.float32); order = presort_features(X32)
    log_w = np.full(len(y), -np.log(len(y)))
    stumps, alphas, errors = [], [], []
    for m in range(n_estimators):
        w = np.exp(log_w - log_w.max()); w = w / w.sum()       # safe: the largest term is exp(0) = 1
        stump = fit_stump(X32, order, y_idx, w, K)
        wrong = predict_stump(stump, X32) != y_idx
        eps = np.sum(w[wrong])
        if eps <= 0 or eps >= 1 - 1 / K:
            break
        alpha = learning_rate * (np.log((1 - eps) / eps) + np.log(K - 1))
        stumps.append(stump); alphas.append(alpha); errors.append(eps)
        log_w = log_w + alpha * wrong
        log_w = log_w - (log_w.max() + np.log(np.sum(np.exp(log_w - log_w.max()))))   # log-sum-exp normalisation
    return {"classes": classes, "stumps": stumps, "alphas": np.array(alphas), "errors": np.array(errors)}


sk_2000 = sklearn_adaboost(2000).fit(Xtr, ytr)
sk_thr_2000 = np.array([e.tree_.threshold[0] for e in sk_2000.estimators_])
for name, model in [("normalised + clipping (sklearn's way)", fit_adaboost_samme(Xtr, ytr, 2000)),
                    ("normalised, no clipping", fit_adaboost_samme(Xtr, ytr, 2000, clip_tiny_weights=False)),
                    ("log-space weights", fit_adaboost_samme_logspace(Xtr, ytr, 2000))]:
    thr = np.array([s["threshold"] for s in model["stumps"]])
    n = min(len(thr), len(sk_thr_2000))
    first = np.flatnonzero(thr[:n] != sk_thr_2000[:n])
    print(f"{name:38s}: {len(thr)} stumps; first stump different from sklearn's: "
          f"{'none' if len(first) == 0 else 'round ' + str(first[0] + 1)};  max |alpha diff| = {np.abs(model['alphas'][:n] - sk_2000.estimator_weights_[:n]).max():.1e}")
w_last = weights_used_by_sklearn(sk_2000, Xtr, ytr)
print(f"weights at machine epsilon (clipped) in scikit-learn's last round: {np.sum(w_last[-1] <= np.finfo(float).eps)} of {len(ytr)}")

normalised + clipping (sklearn's way) : 2000 stumps; first stump different from sklearn's: none;  max |alpha diff| = 0.0e+00
normalised, no clipping               : 2000 stumps; first stump different from sklearn's: round 1612;  max |alpha diff| = 1.4e-14
log-space weights                     : 2000 stumps; first stump different from sklearn's: round 1536;  max |alpha diff| = 1.5e-14
weights at machine epsilon (clipped) in scikit-learn's last round: 20 of 700


> **Implementation hurdle — what this shows**
> * Without normalising, the largest weight grows by orders of magnitude (8.2e+28 after 1000 rounds, 4.4e+189 after 10 000)
>   until the sum overflows: in round 16 574 ε becomes `nan` and the model is garbage from then on. NumPy only *warns*
>   (overflow, invalid value) — nothing stops the run. The relative weights also span more and more orders of magnitude, so
>   cumulative sums lose the small ones (the reason for the `isnan` guard in `split_costs_for_feature`).
> * The textbook update without normalising shrinks the total weight by exactly the training-error bound of README §4.4:
>   after 400 rounds the sum is 0.052061 and so is ∏ 2√(ε(1−ε)) (largest difference 3.3e-16). It would underflow eventually.
> * Normalising every round fixes overflow, but on the clean data some easy points still end up below 2.2e-16 (20 of 700 are
>   clipped in round 2000). Our run **without** clipping, and the log-space run, agree with scikit-learn's α to 1e-14 for all 2000
>   rounds, yet from round 1612 (log space: 1536) they sometimes pick a different stump: two cuts that differ only in where
>   they put points of weight ~1e-16 are a near-tie, and rounding decides it. With scikit-learn's clipping the match is exact
>   for all 2000 rounds.
> * So: always normalise (or work in log space); to reproduce scikit-learn bit for bit, also copy its clipping.

---
## 4 · Decision function and predictions (README §6.3)

scikit-learn's `decision_function` for K classes gives every class a score: each learner adds +α to the class it predicts
and −α/(K − 1) to every other class; then everything is divided by Σα. For K = 2 it returns one number, score(class 1) −
score(class 0) = 2·Σ α_m h_m(x) / Σ α_m with h = ±1, between −2 and +2. The predicted class is the one with the highest score
(for K = 2: class 1 if the number is > 0).

In [15]:
def decision_function_samme(model, X):
    """scikit-learn's normalised SAMME score: (n, K) array, or (n,) for two classes."""
    X32 = X.astype(np.float32)
    K = len(model["classes"])
    votes = sum(np.where(predict_stump(s, X32)[:, None] == np.arange(K), a, -a / (K - 1))
                for s, a in zip(model["stumps"], model["alphas"]))
    votes = votes / np.sum(model["alphas"])
    if K == 2:
        return votes[:, 1] - votes[:, 0]
    return votes


def predict_samme(model, X):
    scores = decision_function_samme(model, X)
    if scores.ndim == 1:
        return model["classes"][(scores > 0).astype(int)]
    return model["classes"][np.argmax(scores, axis=1)]


print("hand example, ours   :", decision_function_samme(samme_hand, x_hand).round(4))
print("hand example, sklearn:", sk_hand.decision_function(x_hand).round(4))
print("2 F / sum(alpha_textbook):", (2 * F_hand / hand["alphas"].sum()).round(4))
assert np.array_equal(decision_function_samme(samme_hand, x_hand), sk_hand.decision_function(x_hand))
assert np.array_equal(predict_samme(samme_hand, x_hand), sk_hand.predict(x_hand))

hand example, ours   : [ 0.947  0.947  0.947  0.947 -0.587 -0.587  0.466 -0.947]
hand example, sklearn: [ 0.947  0.947  0.947  0.947 -0.587 -0.587  0.466 -0.947]
2 F / sum(alpha_textbook): [ 0.947  0.947  0.947  0.947 -0.587 -0.587  0.466 -0.947]


---
## 5 · Comparison with scikit-learn on both CSVs

Same data, same split, same hyper-parameters (`n_estimators`, `learning_rate`, stumps, `random_state=0`). For each run we
report: test accuracy of both, the share of identical test predictions, the largest differences in `estimator_weights_`,
`estimator_errors_` and `decision_function`, how many of the stumps are identical, and the fit times.

In [16]:
def compare_with_sklearn(X_train, y_train, X_test, y_test, n_estimators, learning_rate):
    t0 = time.perf_counter(); ours = fit_adaboost_samme(X_train, y_train, n_estimators, learning_rate); t_ours = time.perf_counter() - t0
    t0 = time.perf_counter(); lib = sklearn_adaboost(n_estimators, learning_rate).fit(X_train, y_train); t_lib = time.perf_counter() - t0
    M = len(lib.estimators_)
    same_stumps = sum(s["feature"] == e.tree_.feature[0] and s["threshold"] == e.tree_.threshold[0]
                      for s, e in zip(ours["stumps"], lib.estimators_))
    return {"acc_ours": np.mean(predict_samme(ours, X_test) == y_test), "acc_lib": lib.score(X_test, y_test),
            "same_pred": np.mean(predict_samme(ours, X_test) == lib.predict(X_test)),
            "d_alpha": np.abs(ours["alphas"] - lib.estimator_weights_[:M]).max(),
            "d_error": np.abs(ours["errors"] - lib.estimator_errors_[:M]).max(),
            "d_decision": np.abs(decision_function_samme(ours, X_test) - lib.decision_function(X_test)).max(),
            "stumps": f"{same_stumps}/{M}", "t_ours": t_ours, "t_lib": t_lib}


print(f"{'data':9s} {'M':>5s} {'eta':>4s} | acc ours  acc sklearn  same pred | max|d alpha| max|d eps| max|d decision| same stumps | fit ours  fit sklearn")
for name in ["good_fit", "bad_fit"]:
    for n_est, lr in [(400, 1.0), (400, 0.5), (2000, 1.0)]:
        r = compare_with_sklearn(*datasets[name], n_est, lr)
        print(f"{name:9s} {n_est:5d} {lr:4.1f} | {r['acc_ours']:.3f}     {r['acc_lib']:.3f}        {r['same_pred']:.1%}    | "
              f"{r['d_alpha']:.1e}      {r['d_error']:.1e}    {r['d_decision']:.1e}        {r['stumps']:>9s} | {r['t_ours']:.2f}s     {r['t_lib']:.2f}s")

data          M  eta | acc ours  acc sklearn  same pred | max|d alpha| max|d eps| max|d decision| same stumps | fit ours  fit sklearn


good_fit    400  1.0 | 0.970     0.970        100.0%    | 0.0e+00      0.0e+00    0.0e+00          400/400 | 0.04s     0.18s


good_fit    400  0.5 | 0.963     0.963        100.0%    | 0.0e+00      0.0e+00    0.0e+00          400/400 | 0.04s     0.18s


good_fit   2000  1.0 | 0.973     0.973        100.0%    | 0.0e+00      0.0e+00    0.0e+00        2000/2000 | 0.21s     0.89s


bad_fit     400  1.0 | 0.753     0.753        100.0%    | 0.0e+00      0.0e+00    0.0e+00          400/400 | 0.04s     0.19s


bad_fit     400  0.5 | 0.763     0.763        100.0%    | 0.0e+00      0.0e+00    0.0e+00          400/400 | 0.04s     0.18s


bad_fit    2000  1.0 | 0.757     0.757        100.0%    | 0.0e+00      0.0e+00    0.0e+00        2000/2000 | 0.20s     0.89s


And three more checks: a **3-class** problem (the rings of the lab's Lab 9), the **textbook** ½-α version against SAMME on
both CSVs, and the **min-error** stump rule from hurdle 3 as a complete model.

In [17]:
rng3 = np.random.default_rng(3)
X3 = np.round(rng3.uniform(-1, 1, size=(1500, 2)), 3)
y3 = np.digitize(np.sqrt((X3 ** 2).sum(axis=1)), [0.45, 0.8])
X3tr, X3te, y3tr, y3te = train_test_split(X3, y3, test_size=0.3, random_state=0, stratify=y3)
r3 = compare_with_sklearn(X3tr, y3tr, X3te, y3te, 400, 1.0)
print(f"3 classes, 400 stumps: acc ours {r3['acc_ours']:.3f} / sklearn {r3['acc_lib']:.3f}, same predictions {r3['same_pred']:.1%}, "
      f"max|d alpha| {r3['d_alpha']:.1e}, max|d decision| {r3['d_decision']:.1e}, same stumps {r3['stumps']}")

for name in ["good_fit", "bad_fit"]:
    X_a, y_a, X_b, y_b = datasets[name]
    textbook = fit_adaboost_textbook(X_a, y_a, 400)
    samme = fit_adaboost_samme(X_a, y_a, 400)
    by_error = fit_adaboost_samme(X_a, y_a, 400, split_finder=find_best_split_by_error)
    n = len(textbook["alphas"])
    print(f"{name}: textbook vs SAMME -> {n} rounds, max |2*alpha_textbook - alpha_SAMME| = "
          f"{np.abs(2 * textbook['alphas'] - samme['alphas'][:n]).max():.1e}, identical test predictions "
          f"{np.mean(predict_textbook(textbook, X_b) == predict_samme(samme, X_b)):.1%};  "
          f"test accuracy with Gini stumps {np.mean(predict_samme(samme, X_b) == y_b):.3f} vs min-error stumps "
          f"{np.mean(predict_samme(by_error, X_b) == y_b):.3f}")

3 classes, 400 stumps: acc ours 0.829 / sklearn 0.829, same predictions 100.0%, max|d alpha| 0.0e+00, max|d decision| 0.0e+00, same stumps 400/400
good_fit: textbook vs SAMME -> 400 rounds, max |2*alpha_textbook - alpha_SAMME| = 1.8e-15, identical test predictions 100.0%;  test accuracy with Gini stumps 0.970 vs min-error stumps 0.973


bad_fit: textbook vs SAMME -> 400 rounds, max |2*alpha_textbook - alpha_SAMME| = 1.7e-15, identical test predictions 100.0%;  test accuracy with Gini stumps 0.753 vs min-error stumps 0.740


> **How to read this**
> * **Exact match.** On both CSVs and all three settings, our NumPy AdaBoost and scikit-learn make **100 % identical test
>   predictions**, every stump is the same (feature and threshold, bit for bit), and the largest differences in
>   `estimator_weights_`, `estimator_errors_` and `decision_function` are **0.0** — not just ~1e-12 but the last bit, because
>   every sum is done in the same order as in scikit-learn. The 3-class problem (400 stumps) matches the same way.
> * **Textbook = SAMME for two classes.** Our textbook version has α exactly half of SAMME's (largest difference 1.8e-15, i.e.
>   rounding) and makes the same test predictions (100 %), as README §4.6 says it must.
> * **Gini vs min-error stumps** both give good models (good fit 0.970 vs 0.973, bad fit 0.753 vs 0.740); they are simply
>   different models.
> * **Speed.** Our pure-NumPy fit is faster here (about 0.04 s vs 0.19 s for 400 rounds, 0.2 s vs 0.9 s for 2000) because it
>   sorts the two features only once for the
>   whole run, while scikit-learn builds each stump as a general tree (sorting again, plus input checks, every round). On
>   bigger data with many features scikit-learn's compiled code would win; the pre-sort idea is what makes ours competitive.

---
# Summary — what each section showed

| Section | Claim from the README | Verdict |
|---|---|---|
| 1 | The weighted stump picks the cut with the lowest weighted Gini; it equals `DecisionTreeClassifier(max_depth=1)` with `sample_weight` | ✔ 50/50 identical |
| 1, hurdle 1 | Sort + cumulative sums make the split search fast; pre-sorting once helps boosting | ✔ timed |
| 1, hurdle 2 | Cuts only between distinct values; float32 midpoints | ✔ shown |
| 1, hurdle 3 | Gini choice ≠ minimum weighted error on some rounds | ✔ shown |
| 2 | README §5 hand example: ε = 1/8, 1/7, 5/24; α = ½ ln 7, ½ ln 6, ½ ln(19/5) | ✔ to 1e-15 |
| 3 | SAMME's α is 2× the textbook α with identical normalised weights | ✔ |
| 3, hurdle 4 | ε = 0 keeps one learner; ε ≥ 1 − 1/K cannot start | ✔ same as scikit-learn |
| 3, hurdle 5 | Without normalising, weights overflow; tiny weights need clipping to match scikit-learn | ✔ shown |
| 4 | `decision_function` = Σ α·(+1 or −1/(K−1)) / Σ α | ✔ identical |
| 5 | NumPy AdaBoost = `AdaBoostClassifier` on both CSVs (and 3 classes) | ✔ exact |